In [2]:
import scipy.io as sio
import numpy as np

# Load subject 1, exercise 1
data = sio.loadmat('data/DB5_s1/S1_E1_A1.mat')
print(data.keys())  # see what's inside

emg = data['emg']        # shape: (samples, 16 channels)
labels = data['restimulus']  # gesture label per sample
print(emg.shape, labels.shape)

dict_keys(['__header__', '__version__', '__globals__', 'emg', 'acc', 'stimulus', 'glove', 'subject', 'exercise', 'repetition', 'restimulus', 'rerepetition', 'age', 'circumference', 'frequency', 'gender', 'height', 'weight', 'laterality', 'sensor'])
(130267, 16) (130267, 1)


In [4]:
# In your notebook — retrain on subject 1 only, all repetitions
import scipy.io as sio
import numpy as np
import pickle
from scipy.signal import butter, filtfilt
from sklearn.ensemble import RandomForestClassifier

def bandpass_filter(signal, lowcut=20, highcut=90, fs=200, order=4):
    nyq = fs / 2
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, np.array(signal), axis=0)

def extract_features(window):
    window = np.array(window)
    mav = np.mean(np.abs(window), axis=0)
    rms = np.sqrt(np.mean(window**2, axis=0))
    zc = np.sum(np.diff(np.sign(window), axis=0) != 0, axis=0)
    wl = np.sum(np.abs(np.diff(window, axis=0)), axis=0)
    return np.concatenate([mav, rms, zc, wl])

def build_dataset(emg, labels, window_size=200, step=50):  # step=50 for more samples
    X, y = [], []
    filtered = bandpass_filter(emg)
    for i in range(0, len(filtered) - window_size, step):
        window = filtered[i:i+window_size]
        label = labels[i + window_size//2][0]
        if label == 0 or label > 6:
            continue
        X.append(extract_features(window))
        y.append(label)
    return np.array(X), np.array(y)

# Load all 3 exercises for subject 1
all_X, all_y = [], []
for ex in ['E1', 'E2', 'E3']:
    try:
        mat = sio.loadmat(f'data/DB5_s1/S1_{ex}_A1.mat')
        emg = mat['emg']
        labels = mat['restimulus']
        X, y = build_dataset(emg, labels)
        # only keep gestures 1-6
        mask = y <= 6
        all_X.append(X[mask])
        all_y.append(y[mask])
        print(f"{ex}: {X[mask].shape}")
    except Exception as e:
        print(f"{ex} skipped: {e}")

X_all = np.concatenate(all_X)
y_all = np.concatenate(all_y)
print(f"Total dataset: {X_all.shape}, labels: {np.unique(y_all)}")

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, min_samples_leaf=2)
model.fit(X_train, y_train)
print(f"Accuracy: {accuracy_score(y_test, model.predict(X_test)):.2%}")

with open('model/gesture_classifier.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Saved.")

E1: (520, 64)
E2: (536, 64)
E3: (540, 64)
Total dataset: (1596, 64), labels: [1 2 3 4 5 6]
Accuracy: 89.69%
Saved.
